# 🛠️ Steg 1: Bygg upp verkstaden
Innan vi kan börja träna vår AI behöver vi hämta verktygen som behövs. Tänk på det som att installera appar på en ny dator. Vi installerar bland annat **Unsloth**, en supermotor som hjälper oss att träna AI:n.

Den här notebooken är anpassad för en egen Linux-server. Installera helst biblioteken i serverns virtuella miljö (`venv` eller `conda`) från terminalen innan du startar notebooken. Då använder notebooken samma Python-miljö varje gång.

**Gör så här:** Om du behöver installera från notebooken, kör kodcellen en gång och vänta tills installationen är klar. Det kan dyka upp mycket text, och ibland kan texten vara röd. Oroa dig inte direkt: röd text är inte alltid ett fel, utan kan vara en varning eller information från datorn. Starta sedan om notebook-kärnan om VS Code eller JupyterLab ber om det.

När du är klar, klicka på **▶ Run** för att köra installationscellen.

In [1]:
# --- Steg 1: Installera bibliotek för träning ---
# På en Linux-server är det bäst att köra installationen i projektets aktiva venv eller conda-miljö.
%pip install unsloth
%pip install --no-deps xformers trl peft loralib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 🎛️ Steg 2: Ställ in din träning
Innan vi startar motorn måste vi berätta för programmet vad vi vill göra. Här har vi samlat de viktigaste inställningarna på ett och samma ställe.

* **`csv_fil`**: Detta är läroboken vi ger till vår AI. Ändra texten inom citattecknen om du vill träna på ett annat dataset, till exempel `"dataset/min_skola.csv"`.
* **`ai_modell`**: Detta är grundhjärnan som redan kan förstå och skriva text. Standardvalet är `unsloth/llama-3-8b-Instruct-bnb-4bit`.
* **`antal_traningssteg`**: Hur många träningssteg AI:n ska få. 30 är en bra start för ett litet dataset.

### 🤖 Varför väljer vi `llama-3-8b-Instruct-bnb-4bit`?

* **Llama 3:** En etablerad språkmodell som redan har lärt sig mycket om mänskligt språk.
* **8B:** Modellen har ungefär 8 miljarder parametrar, alltså många små kopplingar som hjälper den att förstå sammanhang.
* **Instruct:** Modellen är tränad för att följa instruktioner och svara på frågor, vilket passar vår skolbot.
* **bnb-4bit:** Modellen laddas i ett minnessnålt 4-bitarsformat. Det gör att den kan köras och tränas på en server med begränsat VRAM.

Du kan välja en annan kompatibel modell i `ai_modell`, men då behöver modellen fungera med Unsloth och ha stöd för den här typen av träning.

**Gör så här:** Leta upp raden `ai_modell = "unsloth/llama-3-8b-Instruct-bnb-4bit"` i kodcellen nedan om du vill byta grundmodell. Resten av notebooken använder inställningarna automatiskt.

När du är klar, klicka på **▶ Run** för att köra inställningscellen.

In [2]:
# --- Steg 2: Dina inställningar ---
# Ändra värdena här för att styra din träning.
from pathlib import Path

# 1. Vilken lärobok (dataset) ska AI:n läsa?
# Se till att filen ligger i mappen 'dataset'.
csv_fil = "./dataset/gym_och_halsa.csv"

# 2. Vilken bashjärna (AI-modell) ska vi använda?
ai_modell = "unsloth/llama-3-8b-Instruct-bnb-4bit"

# 3. Hur länge ska AI:n sitta i skolbänken?
# Fler steg ger mer träning men tar längre tid.
antal_traningssteg = 30

if not Path(csv_fil).is_file():
    raise FileNotFoundError(f"CSV-filen hittades inte: '{csv_fil}'")

print(f"Valt dataset: {csv_fil}")
print(f"Vald basmodell: {ai_modell}")
print(f"Antal träningssteg: {antal_traningssteg}")

Valt dataset: ./dataset/gym_och_halsa.csv
Vald basmodell: unsloth/llama-3-8b-Instruct-bnb-4bit
Antal träningssteg: 30


# 🧠 Steg 3: Dags för hjärngympa (Träningen börjar!)

Nu ska vi ladda in basmodellen och träna den på våra egna frågor och svar.

### 🤖 Vilken modell använder vi och varför?
Koden laddar automatiskt in den modell du valde i **Steg 2**. Som standard använder vi Llama 3 8B, men du kan välja en annan kompatibel modell i inställningscellen.

* **Miljarder parametrar:** Tänk på detta som att AI:n har miljarder små kopplingar i sin hjärna. Det ger den förkunskap och hjälper den att förstå mänskligt språk.
* **4-bit (krympt storlek):** Vi laddar in en komprimerad version som använder mindre grafikminne. Det gör att modellen kan tränas på servern utan att den behöver bära hela originalstorleken i minnet.
* **Serverns grafikminne (VRAM):** Koden känner automatiskt av vilket grafikkort servern har. Om servern har mycket minne ökas träningspaketen för snabbare körning. Har servern mindre minne används mindre paket så att grafikminnet inte blir fullt.

---

### 📝 Hur fungerar träningen? (Post-it-metoden)
Att skriva om alla miljarder kopplingar skulle ta väldigt lång tid. Därför använder vi en smart teknik som heter **LoRA**:

> I stället för att skriva om hela läroboken klistrar vi små gula **Post-it-lappar** på de viktigaste sidorna. Vi tränar bara det som står på lapparna, ungefär 0,5 % av modellen. Det går mycket snabbare och AI:n behåller det mesta den redan kan.

---

### ⚙️ Träningsinställningarna från Steg 2 och träningsmotorn

* **`antal_traningssteg`:** Det antal uppdateringar, eller tankepauser, AI:n får göra. Fler steg ger mer träning men tar längre tid.
* **Batch size och gradient accumulation:** Koden räknar ut detta automatiskt baserat på serverns kapacitet. Det bestämmer hur många frågor AI:n läser innan den sätter dit en ny Post-it-lapp.
* **Inlärningshastighet (`learning_rate = 1e-4`):** Detta är en intern inställning i träningsmotorn, inte något du behöver ändra i Steg 2. Den bestämmer hur stora kliv AI:n tar när den lär sig.
  * *För högt:* AI:n tar för stora kliv och riskerar att glömma sådant den redan kunde.
  * *För lågt:* AI:n tar så små kliv att den knappt hinner lära sig något.

Klicka på **▶ Run** för att starta träningen och låt servern jobba en stund.

In [3]:
# --- Steg 3: Förbered datat och starta träningen ---
# Ändra inställningar i Steg 2. Låt resten av den här motorn vara som den är.
from datetime import datetime
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

# Läs in CSV-filen från inställningscellen.
df_skolbot = pd.read_csv(csv_fil, encoding="utf-8")

# Skapa en unik sessionmapp så att varje körning sparas separat.
dataset_name = Path(csv_fil).stem
session_name = f"{dataset_name}_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
training_root = Path("training") / session_name
checkpoint_output_dir = training_root / "checkpoints"
model_output_dir = training_root / "model"
gguf_staging_dir = training_root / "gguf_staging"
gguf_export_dir = training_root / "gguf_export"
training_root.mkdir(parents=True, exist_ok=False)

# Läs av serverns GPU och anpassa träningspaketen efter mängden VRAM.
if not torch.cuda.is_available():
    raise RuntimeError("Ingen CUDA-kompatibel GPU hittades. Kontrollera serverns NVIDIA-drivrutin och PyTorch-installation.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
gpu_name = torch.cuda.get_device_name(0)

if vram_gb >= 20:
    batch_size = 4
    grad_accum = 2
else:
    batch_size = 2
    grad_accum = 4

print(f"Server-GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"Anpassad batch size: {batch_size} | Gradient accumulation: {grad_accum}")

# Mall för hur varje exempel ska se ut i modellen.
prompt_template = """Nedan är en fråga från en elev. Skriv ett kort och korrekt svar.

### Fråga:
{}

### Svar:
{}"""

# Ladda den basmodell som valdes i Steg 2.
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ai_modell,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Använd modellens egna slutmarkör så att koden fungerar med andra modeller.
eos_token = tokenizer.eos_token

# Skapa en textkolumn som modellen tränas på.
texts = []
for index, row in df_skolbot.iterrows():
    formatted_text = prompt_template.format(row["Instruction"], row["Response"]) + eos_token
    texts.append(formatted_text)

# Skapa ett Hugging Face-dataset från listan med text.
dataset_dict = {"text": texts}
dataset = Dataset.from_dict(dataset_dict)

# Förbered LoRA-finjning för att göra modellen anpassad till skolfrågor.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# Starta träningsmotorn. Ändra inställningar i Steg 2, inte här.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = batch_size,
        gradient_accumulation_steps = grad_accum,
        warmup_steps = 5,
        max_steps = antal_traningssteg,
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = str(checkpoint_output_dir),
    ),
)

# Starta träningen.
trainer_stats = trainer.train()
model.save_pretrained(str(model_output_dir))
tokenizer.save_pretrained(str(model_output_dir))
print(f"Träningen är slutförd! Resultaten sparas i '{training_root}'.")

/home/karim/Github/unsloth-finetune/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Server-GPU: NVIDIA GeForce RTX 3050 (7.6 GB VRAM)
Anpassad batch size: 2 | Gradient accumulation: 4
==((====))==  Unsloth 2026.9.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3050. Num GPUs = 1. Max memory: 7.636 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 672.95it/s]
Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: Tokenizing ["text"]: 100%|██████████| 70/70 [00:00<00:00, 6632.96 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 70 | Num Epochs = 4 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/home/karim/Github/unsloth-finetune/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/home/karim/Github/unsloth-finetune/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/home/karim/Github/unsloth-finetune/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'to

Step,Training Loss
1,2.553962
2,2.671129
3,2.749581
4,2.753793
5,2.484311
6,2.398524
7,2.337423
8,2.122690
9,1.868785
10,1.693863


Unsloth: Restored added_tokens_decoder metadata in training/gym_och_halsa_2026-09-18_10-30-49/checkpoints/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in training/gym_och_halsa_2026-09-18_10-30-49/model/tokenizer_config.json.


Träningen är slutförd! Resultaten sparas i 'training/gym_och_halsa_2026-09-18_10-30-49'.


# 📦 Steg 4: Exportera modellen som GGUF

Grattis, din AI är färdigtränad! Om du vill köra den på en vanlig laptop, eller i program som LM Studio och Ollama, behöver vi förbereda den.

På en egen Linux-server sparas den nedladdade basmodellen permanent i Hugging Face-cachen, vanligtvis `~/.cache/huggingface/hub`. Första körningen hämtar modellen från internet. Senare körningar kan läsa den från serverns SSD, så samma stora modell behöver normalt inte laddas ner igen.

### 🗜️ Vad är GGUF och kvantisering?

* **GGUF:** Ett särskilt filformat för AI-modeller. Tänk på det som att spara en tung videofil i ett praktiskt format som många olika program kan spela upp.
* **Kvantisering (`q4_k_m`):** Modellens komplicerade decimaltal rundas av och lagras med färre bitar. Filen blir mycket mindre, ungefär fyra gånger mindre än en motsvarande 16-bitarsmodell, samtidigt som kvaliteten oftast fortfarande känns bra.

Koden nedan gör om modellen till en mer lättanvänd version. Det kan ta en stund och datorn kan visa mycket text medan den arbetar. Vänta tills cellen är klar.

### Snabböversikt

| Begrepp i koden | Enkel liknelse |
|---|---|
| Llama 3 8B | Grundhjärnan med miljarder kopplingar. |
| LoRA (PEFT) | Post-it-lappar som klistras ovanpå grundmodellen. |
| Batch size (8) | AI:n pluggar i grupper om 8 frågor innan den gör en justering. |
| Steps (30) | Antalet tankepauser och justeringar under lektionen. |
| Epoker (ungefär 5) | Antalet gånger AI:n hinner läsa igenom frågelistan. |
| GGUF och kvantisering | Att packa modellen och runda av siffrorna så att den ryms på en vanlig dator. |

När du är klar, klicka på **▶ Run** för att köra GGUF-exporten.

In [6]:
# --- Steg 4a: Spara modellen som GGUF ---
# Här exporteras den tränade modellen i GGUF-format med kvantisering för att minska storleken.
import os
import shutil
import stat
from pathlib import Path

if "model" not in globals() or "tokenizer" not in globals():
    print("GGUF-exporten kan inte starta ännu.")
    print("Kör träningscellen först så att variablerna 'model' och 'tokenizer' skapas.")
else:
    exported_gguf_path = gguf_export_dir / f"{dataset_name}_finetuned_q4_k_m.gguf"

    if gguf_export_dir.exists():
        # Gör cellen återkörbar om exportmappen redan skapades vid en tidigare körning.
        gguf_files = list(gguf_export_dir.glob("*.gguf"))
        if len(gguf_files) != 1:
            raise FileNotFoundError(f"Förväntade exakt en GGUF-fil i {gguf_export_dir}, hittade {len(gguf_files)}.")
        if gguf_files[0] != exported_gguf_path:
            if exported_gguf_path.exists():
                raise FileExistsError(f"Målfilen finns redan: {exported_gguf_path}")
            gguf_files[0].rename(exported_gguf_path)
        print(f"GGUF-exporten finns redan: '{exported_gguf_path}'.")
    else:
        os.makedirs(gguf_staging_dir, exist_ok=True)

        # Unsloth mergar shards på plats. Gör cachekopiorna skrivbara först.
        cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
        for shard_path in cache_dir.rglob("model-*-of-00004.safetensors"):
            shard_path.chmod(shard_path.stat().st_mode | stat.S_IWUSR)
        for shard_path in Path(gguf_staging_dir).glob("model-*.safetensors"):
            shard_path.chmod(shard_path.stat().st_mode | stat.S_IWUSR)

        model.save_pretrained_gguf(
            gguf_staging_dir,
            tokenizer,
            quantization_method = "q4_k_m",
        )

        # Unsloth lägger automatiskt till suffixet '_gguf'. Ge slutmappen ett tydligt namn.
        generated_gguf_dir = Path(f"{gguf_staging_dir}_gguf")
        if not generated_gguf_dir.exists():
            raise FileNotFoundError(f"Hittade inte den exporterade GGUF-mappen: {generated_gguf_dir}")

        generated_gguf_dir.rename(gguf_export_dir)
        if gguf_staging_dir.exists():
            shutil.rmtree(gguf_staging_dir)

        gguf_files = list(gguf_export_dir.glob("*.gguf"))
        if len(gguf_files) != 1:
            raise FileNotFoundError(f"Förväntade exakt en GGUF-fil i {gguf_export_dir}, hittade {len(gguf_files)}.")
        gguf_files[0].rename(exported_gguf_path)
        print(f"Modellen har exporterats till '{exported_gguf_path}'.")

GGUF-exporten finns redan: 'training/gym_och_halsa_2026-09-18_10-30-49/gguf_export/gym_och_halsa_finetuned_q4_k_m.gguf'.


# 📂 Steg 5: Var hamnade filen?
Den här cellen visar vilken arbetsmapp notebooken använder. Det hjälper oss att hitta träningsresultatet på servern.

Varje gång du kör träningen skapas en **ny sessionsmapp**. Den får ett namn som till exempel `ai_intro_2026-09-18_14-30-05`, så att nya resultat inte skriver över gamla.

Öppna projektmappen i VS Codes filträd eller i JupyterLabs filbläddrare. Gå till `training` och sedan din sessionsmapp. Där hittar du mappen `gguf_export` med den färdiga GGUF-filen, tillsammans med mapparna `model` och `checkpoints`.

På servern kan du också hitta filerna direkt från terminalen med sökvägen `training/<session_name>/`. För LM Studio eller Ollama behöver du normalt bara kopiera GGUF-filen från `gguf_export`.

När du är klar, klicka på **▶ Run** för att kontrollera arbetskatalogen.

In [ ]:
# --- Steg 5: Kontrollera arbetskatalogen ---
# Här ser vi var projektet körs och vilken katalog som innehåller de exporterade modellerna.
import os

print(f"Lokal arbetskatalog: {os.getcwd()}")

# 💬 Steg 6: Testa att prata med din skapelse!
Nu kommer den roligaste delen: att se om träningen fungerade.

I koden nedan finns raden:

`test_fraga = "Vad är AI egentligen?"`

**Gör så här:**
1. Byt ut frågan inom citattecknen mot något som passar ditt dataset, till exempel `Vad är maskininlärning?` eller `Använder TikTok AI?`.
2. Läs AI:ns svar och fundera på vad den gjorde bra och vad som kanske kan förbättras.

Kom ihåg: AI:n gissar utifrån de exempel den fick under träningen. Ett konstigt svar betyder inte att du har gjort fel; det är en ledtråd om vad som kan tränas bättre nästa gång.

När du är klar, klicka på **▶ Run** för att testa modellen.

In [ ]:
# --- Steg 6: Testa modellen med exempelfrågor ---
# Här låter vi modellen svara på några frågor för att kontrollera att träningen fungerade som tänkt.
import sys
import torch

try:
    from unsloth import FastLanguageModel
except ModuleNotFoundError:
    print("FEL: Unsloth saknas i denna aktiva session!")
    print("Det betyder att Python-miljön inte har fått installationscellen körd ännu.")
    print("LÖSNING: Kör Steg 1 (installera unsloth) och därefter Steg 3 (träningscellen) på nytt.")
    sys.exit()

# Förbered modellen för snabb slutledning (inference)
try:
    FastLanguageModel.for_inference(model)
except NameError:
    print("FEL: Variabeln 'model' hittades inte.")
    print("Kör Steg 3 (träningscellen) en gång för att ladda in modellen i minnet först!")
    sys.exit()

def fraga_skolbot(fraga):
    # Formatera frågan exakt enligt den mall modellen tränades på
    input_text = prompt_template.format(fraga, "")

    inputs = tokenizer([input_text], return_tensors = "pt").to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 64,
            use_cache = True,
            temperature = 0.1,
        )

    # Avkoda svaret och rensa bort instruktionsmallen
    fullt_svar = tokenizer.decode(outputs[0], skip_special_tokens=True)
    svar = fullt_svar.split("### Svar:")[1].strip() if "### Svar:" in fullt_svar else fullt_svar
    return svar

# Exempeltest - ändra gärna till egna frågor!
test_fraga = "Vad är AI egentligen?"
print(f"Fråga: {test_fraga}")
print(f"Svar: {fraga_skolbot(test_fraga)}")